All In One ONNX to FPGA: ONNX → fpgaConvNet → HLS → FPGA

This notebook merges:
- 2.1/2.2/2.3: Model selection, (optional) quantization, ONNX export
- 3.1/3.2/3.3: ONNX parsing, network partition, layer module preparation
- 4.1/4.2/4.3: Optimiser transforms, solver, config/report
- 5: HLS project generation & build

# Imports & Paths

In [1]:
# =========================
# Imports (sorted & grouped)
# =========================

# --- Standard library ---
import os
import json
import pathlib            # keep if you use pathlib.Path elsewhere
from pathlib import Path  # used in Section 5
import types              # used for small monkey-patches (2.1), keep if needed

# --- Third-party ---
import numpy as np        # Section 5
import onnx
import torch
import toml
import netron
from IPython.display import display, JSON  # note: JSON here is a display helper, not the json module

# --- Project / toolflow ---
from models import initialize_wrapper                         # 2.1
from models.classification import utils                       # 2.1
from quantization.utils import QuantMode, quantize_model      # 2.2
from fpgaconvnet.parser.Parser import Parser                  # 3.1
import fpgaconvnet.optimiser.transforms as transforms         # 4.1
from fpgaconvnet.platform.Platform import Platform            # 4.2
from fpgaconvnet.optimiser.solvers import (
    GreedyPartition, SimulatedAnnealing, Solver               # 4.2 / 4.3
)
from fpgaconvnet.hls.generate.network import GenerateNetwork  # 5


# =========================
# Config (grouped, same vals)
# =========================

# --- Dataset ---
dataset_name = 'cifar10'
dataset_path = "ProjectFiles/dataset/cifar10"

# --- Model & board ---
model_name = 'mobilenetv2_x0_5'
board = 'ProjectFiles/platforms/zedboard.toml'

# --- ONNX export paths (DO NOT CHANGE)---
generate_onnx_files_fp16 = 'ProjectFiles/onnx/fp16'
generate_onnx_path_fp16  = f'ProjectFiles/onnx/fp16/{model_name}.onnx'
generate_onnx_files_bfp8 = 'ProjectFiles/onnx/bfp8'
generate_onnx_path_bfp8  = f'ProjectFiles/onnx/bfp8/{model_name}.onnx'

# Choose which ONNX to use downstream
onnx_path = f'ProjectFiles/onnx/fp16/{model_name}.onnx'
#onnx_path = f'ProjectFiles/onnx/bfp8/{model_name}.onnx' # Note: hls exportation not available for bfp8

# --- Reports / configs (DO NOT CHANGE)---
pathlib_sa         = 'ProjectFiles/reports/sa'
pathlib_sa_report  = 'ProjectFiles/reports/sa/report.json'
pathlib_sa_config  = 'ProjectFiles/reports/sa/config.json'
pathlib_gp         = 'ProjectFiles/reports/gp'
pathlib_gp_report  = 'ProjectFiles/reports/gp/report.json'
pathlib_gp_config  = 'ProjectFiles/reports/gp/config.json'

# --- Runtime params ---
batch_size = 64
workers    = 4

# 2.1_torch_model_zoo

We set up the project wrapper (paths, dataset hooks, backbone name, data-loader options) and get back a model_wrapper we’ll reuse.
This helps keep every later step consistent (same model/data) and avoids hidden path or config surprises. To check model compatiblility and naming: https://github.com/chenyaofo/pytorch-cifar-models

In [2]:
# Check model compatibility with cifar-10
from pprint import pprint
pprint(torch.hub.list("chenyaofo/pytorch-cifar-models", force_reload=True))

Downloading: "https://github.com/chenyaofo/pytorch-cifar-models/zipball/master" to /home/ubuntu/.cache/torch/hub/master.zip
['cifar100_mobilenetv2_x0_5',
 'cifar100_mobilenetv2_x0_75',
 'cifar100_mobilenetv2_x1_0',
 'cifar100_mobilenetv2_x1_4',
 'cifar100_repvgg_a0',
 'cifar100_repvgg_a1',
 'cifar100_repvgg_a2',
 'cifar100_resnet20',
 'cifar100_resnet32',
 'cifar100_resnet44',
 'cifar100_resnet56',
 'cifar100_shufflenetv2_x0_5',
 'cifar100_shufflenetv2_x1_0',
 'cifar100_shufflenetv2_x1_5',
 'cifar100_shufflenetv2_x2_0',
 'cifar100_vgg11_bn',
 'cifar100_vgg13_bn',
 'cifar100_vgg16_bn',
 'cifar100_vgg19_bn',
 'cifar100_vit_b16',
 'cifar100_vit_b32',
 'cifar100_vit_h14',
 'cifar100_vit_l16',
 'cifar100_vit_l32',
 'cifar10_mobilenetv2_x0_5',
 'cifar10_mobilenetv2_x0_75',
 'cifar10_mobilenetv2_x1_0',
 'cifar10_mobilenetv2_x1_4',
 'cifar10_repvgg_a0',
 'cifar10_repvgg_a1',
 'cifar10_repvgg_a2',
 'cifar10_resnet20',
 'cifar10_resnet32',
 'cifar10_resnet44',
 'cifar10_resnet56',
 'cifar10_shuf

In [3]:
#For cifar-10 comaptible models
model_wrapper = initialize_wrapper(dataset_name, model_name,
                                    dataset_path, batch_size, workers)

Using cache found in /home/ubuntu/.cache/torch/hub/chenyaofo_pytorch-cifar-models_master


We print/inspect the model architecture to see the backbone and final layers that were created.
This helps confirm we’re using the intended model before profiling or exporting.

In [4]:
model_wrapper.model

MobileNetV2(
  (features): Sequential(
    (0): ConvBNActivation(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): ConvBNActivation(
          (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
          (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(16, 8, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): ConvBNActivation(
          (0): Conv2d(8, 48, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(48, eps=1e-05, momentum

We switch the model to eval(), choose a profiling device, and run a quick profile (e.g., params/FLOPs, basic layer stats).
This helps estimate compute/memory needs early and catch obvious shape or op issues.

In [5]:
model_wrapper.model.eval().to("cpu")   # put model on CPU for tracing
torch.cuda.empty_cache()               # optional, just to tidy up GPU mem

# now profile
model_wrapper.profile()

Unsupported operator aten::hardtanh_ encountered 35 time(s)
Unsupported operator aten::add encountered 10 time(s)


MACs: 27989504, Params: 700490


We pick the best available device (GPU if present, else CPU), move data/model there, and enable a safe no-grad inference helper.
This helps prevent device-mismatch errors and makes runs reproducible across different machines.

In [6]:
# Detect the available device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move model to that device
model_wrapper.model.to(device)

# Patch dataloader targets to same device inside inference
def _inference_fixed(data_loader, model, criterion, print_freq=20, silence=False):
    model.eval()
    with torch.no_grad():
        for i, (images, target) in enumerate(data_loader):
            images = images.to(device)
            target = target.to(device)
            output = model(images)
    print("Inference completed successfully on", device)

utils._inference = _inference_fixed

We run a small sanity inference on a validation/test split to exercise the full data-loader → model path.
This helps verify the pipeline works end-to-end before quantisation and ONNX export.

In [7]:
model_wrapper.inference('test')

Inference mode: test
 * Acc@1 93.110 Acc@5 99.820


(93.11000061035156, 99.81999969482422)

# 2.2_torch_quantization

We select the quantization path (dynamic, post-training static, or QAT), set the backend (e.g., FBGEMM/QNNPACK), and define dtypes (INT8/FP16) and any wrapper flags.
This helps keep a single source of truth for quantization settings so every later step is consistent and reproducible.

In [8]:
model_wrapper = initialize_wrapper(dataset_name, model_name,
                                    dataset_path, batch_size, workers)

Using cache found in /home/ubuntu/.cache/torch/hub/chenyaofo_pytorch-cifar-models_master


We fuse compatible patterns (e.g., Conv + BatchNorm + ReLU) in place before quantization.
This helps the quantizer see simpler ops, usually improving accuracy and performance after quantization.

In [9]:
print("FLOAT32 Inference")
model_wrapper.inference()

FLOAT32 Inference
Inference mode: test
 * Acc@1 93.110 Acc@5 99.820


(93.11000061035156, 99.81999969482422)

We apply dynamic quantization to weight-heavy layers (e.g., Linear/LSTM) and run a quick dry-run to check shapes.
This helps reduce model size and CPU latency with minimal code changes and is often the most export-friendly option.

In [10]:
from quantization.utils import QuantMode, quantize_model

We prepare the model for static quantization, run a short calibration loop over a few batches, then convert to an INT8 model.
This helps the quantizer learn good scales/zero-points from real data, improving accuracy versus uncalibrated INT8.

In [11]:
print("NETWORK FP16 Inference")
# reload the model everytime a new quantization mode is tested
model_wrapper.load_model()
quantize_model(model_wrapper, {
                'weight_width': 16, 'data_width': 16, 'mode': QuantMode.NETWORK_FP})
model_wrapper.inference("test")

NETWORK FP16 Inference
network weight min: tensor(-0.6861, device='cuda:0', grad_fn=<MinimumBackward0>)
network weight max: tensor(0.8963, device='cuda:0', grad_fn=<MaximumBackward0>)
Inference mode: calibrate


Using cache found in /home/ubuntu/.cache/torch/hub/chenyaofo_pytorch-cifar-models_master
/home/ubuntu/fpgaConvNet/fpgaconvnet-torch/quantization/utils.py:96: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:835.)
  w_min = torch.tensor([self.w_min])


 * Acc@1 99.940 Acc@5 100.000
activation min: tensor(-7.3362, device='cuda:0')
activation max: tensor(23.1431, device='cuda:0')
Inference mode: test
 * Acc@1 93.140 Acc@5 99.810


(93.13999938964844, 99.80999755859375)

We enable QAT (fake quantization during training), run a brief fine-tuning phase, then convert to the final quantized model.
This helps recover accuracy on sensitive architectures by letting the model adapt to quantization effects.

In [12]:
print("NETWORK FP8 Inference")
model_wrapper.load_model()
quantize_model(model_wrapper, {
                'weight_width': 8, 'data_width': 8, 'mode': QuantMode.NETWORK_FP})
model_wrapper.inference("test")

NETWORK FP8 Inference
network weight min: tensor(-0.6861, device='cuda:0', grad_fn=<MinimumBackward0>)
network weight max: tensor(0.8963, device='cuda:0', grad_fn=<MaximumBackward0>)
Inference mode: calibrate


Using cache found in /home/ubuntu/.cache/torch/hub/chenyaofo_pytorch-cifar-models_master


 * Acc@1 99.860 Acc@5 100.000
activation min: tensor(-7.3400, device='cuda:0')
activation max: tensor(23.6337, device='cuda:0')
Inference mode: test
 * Acc@1 11.180 Acc@5 51.080


(11.179999351501465, 51.07999801635742)

We evaluate original vs. quantized variants (accuracy, latency, and model size) and record the results.
This helps us pick the best trade-off for the FPGA pipeline and ensures the chosen model is ready for ONNX export in 2.3.

In [13]:
print("CHANNEL BFP8 Inference")
model_wrapper.load_model()
quantize_model(model_wrapper,  {
                'weight_width': 8, 'data_width': 8, 'mode': QuantMode.CHANNEL_BFP})
model_wrapper.inference("test")

CHANNEL BFP8 Inference
network weight min: tensor(-0.6861, device='cuda:0', grad_fn=<MinimumBackward0>)
network weight max: tensor(0.8963, device='cuda:0', grad_fn=<MaximumBackward0>)
Inference mode: calibrate


Using cache found in /home/ubuntu/.cache/torch/hub/chenyaofo_pytorch-cifar-models_master


 * Acc@1 99.940 Acc@5 100.000
activation min: tensor(-7.3320, device='cuda:0')
activation max: tensor(23.0304, device='cuda:0')
Inference mode: test
 * Acc@1 92.930 Acc@5 99.800


(92.93000030517578, 99.79999542236328)

# 2.3_torch_onnx_export

We choose the export settings (opset, precision FP32/FP16, input/output names, output paths) and prepare the model for export (eval(), correct device) along with a correctly shaped dummy input.
This helps ensure the trace is deterministic and the exported graph matches the exact I/O contract we’ll use downstream.

In [14]:
model_wrapper = initialize_wrapper(dataset_name, model_name,
                                    dataset_path, batch_size, workers)

Using cache found in /home/ubuntu/.cache/torch/hub/chenyaofo_pytorch-cifar-models_master


We call torch.onnx.export(...) with dynamic batch axes and constant folding to produce the raw .onnx file.
This helps create a portable, framework-agnostic model that tools like parsers and hardware flows can consume reliably.

In [15]:
print("NETWORK FP16 Inference")
# reload the model everytime a new quantization mode is tested
model_wrapper.load_model()
quantize_model(model_wrapper, {
                'weight_width': 16, 'data_width': 16, 'mode': QuantMode.NETWORK_FP})
model_wrapper.inference("test")
model_wrapper.generate_onnx_files(generate_onnx_files_fp16)

Using cache found in /home/ubuntu/.cache/torch/hub/chenyaofo_pytorch-cifar-models_master


NETWORK FP16 Inference
network weight min: tensor(-0.6861, device='cuda:0', grad_fn=<MinimumBackward0>)
network weight max: tensor(0.8963, device='cuda:0', grad_fn=<MaximumBackward0>)
Inference mode: calibrate
 * Acc@1 99.880 Acc@5 100.000
activation min: tensor(-8.1027, device='cuda:0')
activation max: tensor(23.8183, device='cuda:0')
Inference mode: test
 * Acc@1 93.060 Acc@5 99.820


Using cache found in /home/ubuntu/.cache/torch/hub/chenyaofo_pytorch-cifar-models_master


'ProjectFiles/onnx/fp16/mobilenetv2_x0_5.onnx'

We simplify and validate the ONNX (run ONNX-Simplifier, then onnx.checker / shape inference), with a fallback that handles “initializers as inputs” when needed.
This helps reduce graph complexity, avoid common parsing issues, and confirm the model is structurally sound before hardware steps.

In [16]:
netron.start(generate_onnx_path_fp16)

Serving 'ProjectFiles/onnx/fp16/mobilenetv2_x0_5.onnx' at http://localhost:15704


('localhost', 15704)

We sanity-test the ONNX with ONNX Runtime on a dummy batch and print a short summary (paths, opset, precision, I/O shapes).
This helps verify the model runs outside PyTorch and gives a clear snapshot of what was exported before moving to parsing (3.1).

In [17]:
model_wrapper.load_model()
model_wrapper.sideband_info['quantization'] = {
                'weight_width': 8, 'data_width': 8, 'mode': QuantMode.CHANNEL_BFP}
model_wrapper.generate_onnx_files(generate_onnx_files_bfp8)
netron.start(generate_onnx_path_bfp8)

Using cache found in /home/ubuntu/.cache/torch/hub/chenyaofo_pytorch-cifar-models_master
Using cache found in /home/ubuntu/.cache/torch/hub/chenyaofo_pytorch-cifar-models_master
Serving 'ProjectFiles/onnx/bfp8/mobilenetv2_x0_5.onnx' at http://localhost:15647


('localhost', 15647)

# 3.1_model_onnx_parser

We import the parser utilities, set the ONNX input path, choose output locations for IR files, and define basic options (layout/precision).
This helps keep all parser settings in one place so the rest of the steps stay consistent and easy to repeat.

In [18]:
parser = Parser(custom_onnx=True, batch_size=1)
net = parser.onnx_to_fpgaconvnet(onnx_path)

CRITICAL WARNING: node Flatten_98 is skipped in hardware


We load the ONNX graph and quickly inspect inputs/outputs (names, shapes, dtypes).
This helps confirm we’re parsing the exact tensors we expect and prevents shape/name mismatches later.

In [19]:
for i, partition in enumerate(net.partitions):
    print(i, partition)
    for j, layer_name in enumerate(partition.graph.nodes):
        layer_type = partition.graph.nodes[layer_name]["type"]
        layer = partition.graph.nodes[layer_name]["hw"]
        print(j, layer_name, layer_type)

0 <fpgaconvnet.models.partition.Partition.Partition object at 0x72cab1fd7970>
0 Conv_0 LAYER_TYPE.Convolution
1 Relu_1 LAYER_TYPE.ReLU
2 Conv_2 LAYER_TYPE.Convolution
3 Relu_3 LAYER_TYPE.ReLU
4 Conv_4 LAYER_TYPE.Convolution
5 Conv_5 LAYER_TYPE.Convolution
6 Relu_6 LAYER_TYPE.ReLU
7 Conv_7 LAYER_TYPE.Convolution
8 Relu_8 LAYER_TYPE.ReLU
9 Conv_9 LAYER_TYPE.Convolution
10 Conv_10 LAYER_TYPE.Convolution
11 Relu_11 LAYER_TYPE.ReLU
12 Conv_12 LAYER_TYPE.Convolution
13 Relu_13 LAYER_TYPE.ReLU
14 Conv_14 LAYER_TYPE.Convolution
15 Add_15 LAYER_TYPE.EltWise
16 Conv_16 LAYER_TYPE.Convolution
17 Relu_17 LAYER_TYPE.ReLU
18 Conv_18 LAYER_TYPE.Convolution
19 Relu_19 LAYER_TYPE.ReLU
20 Conv_20 LAYER_TYPE.Convolution
21 Conv_21 LAYER_TYPE.Convolution
22 Relu_22 LAYER_TYPE.ReLU
23 Conv_23 LAYER_TYPE.Convolution
24 Relu_24 LAYER_TYPE.ReLU
25 Conv_25 LAYER_TYPE.Convolution
26 Add_26 LAYER_TYPE.EltWise
27 Conv_27 LAYER_TYPE.Convolution
28 Relu_28 LAYER_TYPE.ReLU
29 Conv_29 LAYER_TYPE.Convolution
30 Relu_3

We normalize names, remove unused items, and prepare simple lookups (node → tensor maps) the parser needs.
This helps make parsing more robust and reduces surprises from odd node names or dangling tensors.

In [20]:
net.partitions[0].graph.nodes["Conv_0"].keys()

dict_keys(['type', 'onnx_node', 'onnx_input', 'onnx_output', 'attr', 'hw', 'inputs'])

We run the parser to translate ONNX nodes into the internal IR: layers, tensors, and connections.
This helps move from a framework-agnostic model to a hardware-friendly description the next stages can understand.

In [21]:
net.partitions[0].graph.nodes["Conv_0"]["type"]

<LAYER_TYPE.Convolution: 4>

We check the IR for consistency (acyclic graph, compatible shapes/strides/padding, supported ops) and report a short summary.
This helps catch problems early, before partitioning or optimisation make them harder to debug.

In [22]:
net.partitions[0].graph.nodes["Conv_0"]["hw"]

ConvolutionLayer(_rows=32, _cols=32, _channels=3, _coarse_in=1, _coarse_out=1, input_compression_ratio=[1.0, 1.0, 1.0], output_compression_ratio=[1.0], mem_bw_in=100.0, mem_bw_out=100.0, data_t=FixedPoint(width=16, binary_point=8), buffer_depth=2, modules=OrderedDict([('pad', Pad(rows=32, cols=32, channels=3, data_width=16, rsc_coef={'FF': [], 'LUT': [], 'DSP': [], 'BRAM': []}, pad_top=1, pad_bottom=1, pad_left=1, pad_right=1, backend='chisel', regression_model='linear_regression', streams=1, latency_mode=False, block=False)), ('sliding_window', SlidingWindow(rows=34, cols=34, channels=3, data_width=16, rsc_coef={'Logic_LUT': array([6.46840521, 0.        , 0.        , 3.92659388, 2.45084112]), 'LUT_RAM': array([ 0.07302216,  0.        , 14.23619095]), 'LUT_SR': array([0.]), 'FF': array([ 0.        ,  0.        , 20.77153538,  0.        ,  0.        ,
        0.        ,  5.21236925,  3.44020551,  0.        ,  0.        ]), 'DSP': array([0.]), 'BRAM36': array([0.]), 'BRAM18': array([0.]

We save the IR (e.g., JSON) and any helper indexes/maps for later notebooks.
This helps downstream steps (partition, optimiser, HLS) load the exact same model without re-parsing.

In [23]:
net.partitions[0].graph.nodes["Conv_0"]["hw"].modules

OrderedDict([('pad',
              Pad(rows=32, cols=32, channels=3, data_width=16, rsc_coef={'FF': [], 'LUT': [], 'DSP': [], 'BRAM': []}, pad_top=1, pad_bottom=1, pad_left=1, pad_right=1, backend='chisel', regression_model='linear_regression', streams=1, latency_mode=False, block=False)),
             ('sliding_window',
              SlidingWindow(rows=34, cols=34, channels=3, data_width=16, rsc_coef={'Logic_LUT': array([6.46840521, 0.        , 0.        , 3.92659388, 2.45084112]), 'LUT_RAM': array([ 0.07302216,  0.        , 14.23619095]), 'LUT_SR': array([0.]), 'FF': array([ 0.        ,  0.        , 20.77153538,  0.        ,  0.        ,
                      0.        ,  5.21236925,  3.44020551,  0.        ,  0.        ]), 'DSP': array([0.]), 'BRAM36': array([0.]), 'BRAM18': array([0.])}, kernel_size=[3, 3], stride=[1, 1], pad_top=0, pad_right=0, pad_bottom=0, pad_left=0, backend='chisel', regression_model='linear_regression', streams=1)),
             ('squeeze',
              Sque

# 3.2_model_network_partition

We load the parsed IR and platform settings (device part, clock, port width) and define where to save partition artifacts.
This helps guarantee we partition the exact same model with the exact same board assumptions, keeping results reproducible.

In [24]:
parser = Parser(custom_onnx=True, batch_size=1)
net = parser.onnx_to_fpgaconvnet(onnx_path)

CRITICAL WARNING: node Flatten_98 is skipped in hardware


We choose the partition strategy and constraints (e.g., number of stages, bandwidth/memory limits, objective like latency or throughput).
This helps make the search space explicit so the algorithm knows what “good” means for our hardware.

In [25]:
print("Number of partitions:", len(net.partitions))

multi_device = False
freq = 200 # MHz
reconf_time = 0.08255 # second

cycles = int(net.get_cycle(multi_device))
latency = net.get_latency(freq, multi_device, reconf_time)
throughput = net.get_throughput(freq, multi_device, reconf_time)

print("Cycles: ", cycles)
print("Latency (seconds): ", latency)
print("Throughput (frames per second): ", throughput)

Number of partitions: 1
Cycles:  362192635
Latency (seconds):  1.810963175
Throughput (frames per second):  0.5521923437233891


We run the partition algorithm and assign layers/tensors to stages, then run quick checks on cross-stage interfaces (tensor shapes, rate matching).
This helps split the network into hardware-friendly chunks and prevents boundary mismatches that would break HLS later.

In [26]:
for i, partition in enumerate(net.partitions):
    print(f"{i}, resource: ", partition.get_resource_usage())

0, resource:  {'FF': 76310, 'LUT': 92878, 'DSP': 55, 'BRAM': 1192, 'URAM': 0}


We compute simple per-stage estimates (ops, feature sizes, rough bandwidth) and save the partition plan (JSON + a short summary/figure if available).
This helps spot obvious bottlenecks early and gives downstream steps (optimiser/HLS) a clean, reusable partition description.

In [27]:
parser = Parser(custom_onnx=True, batch_size=1)
net = parser.onnx_to_fpgaconvnet(generate_onnx_path_bfp8)
for i, partition in enumerate(net.partitions):
    print(f"{i}, resource: ", partition.get_resource_usage())

CRITICAL WARNING: node Flatten_98 is skipped in hardware
0, resource:  {'FF': 66333, 'LUT': 69925, 'DSP': 107.5, 'BRAM': 716, 'URAM': 0}


# 3.3_model_layer_module

We load the parsed IR (and partition plan if available) and initialise the layer-module builder with defaults (precision/Q-format, stream/port width, buffer policy).
This helps ensure every module is derived from the same assumptions so later steps stay consistent.

In [28]:
parser = Parser(custom_onnx=True, batch_size=1)
net = parser.onnx_to_fpgaconvnet(onnx_path)
conv_0_layer = net.partitions[0].graph.nodes["Conv_0"]["hw"]
conv_0_layer.modules

CRITICAL WARNING: node Flatten_98 is skipped in hardware


OrderedDict([('pad',
              Pad(rows=32, cols=32, channels=3, data_width=16, rsc_coef={'FF': [], 'LUT': [], 'DSP': [], 'BRAM': []}, pad_top=1, pad_bottom=1, pad_left=1, pad_right=1, backend='chisel', regression_model='linear_regression', streams=1, latency_mode=False, block=False)),
             ('sliding_window',
              SlidingWindow(rows=34, cols=34, channels=3, data_width=16, rsc_coef={'Logic_LUT': array([6.46840521, 0.        , 0.        , 3.92659388, 2.45084112]), 'LUT_RAM': array([ 0.07302216,  0.        , 14.23619095]), 'LUT_SR': array([0.]), 'FF': array([ 0.        ,  0.        , 20.77153538,  0.        ,  0.        ,
                      0.        ,  5.21236925,  3.44020551,  0.        ,  0.        ]), 'DSP': array([0.]), 'BRAM36': array([0.]), 'BRAM18': array([0.])}, kernel_size=[3, 3], stride=[1, 1], pad_top=0, pad_right=0, pad_bottom=0, pad_left=0, backend='chisel', regression_model='linear_regression', streams=1)),
             ('squeeze',
              Sque

We derive raw implementation parameters per layer (tile sizes, folding factors, number of PEs/MACs) and compute expected I/O tensor shapes and rates.
This helps translate abstract layers into concrete hardware candidates that can fit the device.

In [29]:
print("coarse_in: ", conv_0_layer.coarse_in)
print("coarse_out: ", conv_0_layer.coarse_out)
print("coarse_group: ", conv_0_layer.coarse_group)
print("fine: ", conv_0_layer.fine)

coarse_in:  1
coarse_out:  1
coarse_group:  1
fine:  1


We resolve/choose final parameters per layer under simple constraints (resource/bandwidth/latency targets) and lock in numeric formats where needed.
This helps produce a single, coherent configuration per layer that downstream tools can build without guesswork.

In [30]:
print("Latency (cycle):", conv_0_layer.latency())
print(conv_0_layer.resource())

Latency (cycle): 442368
{'LUT': 4140, 'FF': 2094, 'DSP': 1, 'BRAM': 2, 'URAM': 0}


We validate interfaces between adjacent modules (layout, coarse in/out factors, stride/padding alignment, stream width) and apply safe fix-ups if possible.
This helps prevent boundary mismatches that would break stitching, HLS, or cosimulation later.

In [31]:
accum_module = conv_0_layer.modules["accum"]
print("rsc_coef: ", accum_module.rsc_coef)
print("utilisation_model: ", accum_module.utilisation_model())
print("accum rsc: ",accum_module.rsc())

rsc_coef:  {'Logic_LUT': array([0.85353288, 0.        , 3.51035984, 0.        , 0.        ,
       0.        , 0.        ]), 'LUT_RAM': array([13.67379136,  0.01678309,  0.92521122,  0.        ]), 'LUT_SR': array([0.]), 'FF': array([0.        , 3.20979382, 0.        , 0.        , 0.00732597,
       0.28885062, 0.        ]), 'DSP': array([0.]), 'BRAM36': array([0.]), 'BRAM18': array([0.])}
utilisation_model:  {'Logic_LUT': array([16, 27, 32,  1,  5,  4,  1]), 'LUT_RAM': array([  3, 512,  32,  16]), 'LUT_SR': array([0]), 'FF': array([32, 32,  5,  4, 27, 16,  1]), 'DSP': array([0]), 'BRAM36': array([0]), 'BRAM18': array([0])}
accum rsc:  {'Logic_LUT': 125, 'LUT_RAM': 79, 'LUT_SR': 0, 'FF': 107, 'DSP': 0, 'BRAM36': 0, 'BRAM18': 0, 'LUT': 204, 'BRAM': 0}


We emit the artifacts (per-layer JSON/headers/templates and a layers index) and print a compact summary of modules and connections.
This helps make the build reproducible and gives the optimiser/HLS everything needed without re-computing these details.

In [32]:
conv_0_layer.coarse_in = conv_0_layer.get_coarse_in_feasible()[-1]
conv_0_layer.coarse_out = conv_0_layer.get_coarse_out_feasible()[-1]
conv_0_layer.coarse_group = conv_0_layer.get_coarse_group_feasible()[-1]
conv_0_layer.fine = conv_0_layer.get_fine_feasible()[-1]

conv_0_layer.update()

print("coarse_in: ", conv_0_layer.coarse_in)
print("coarse_out: ", conv_0_layer.coarse_out)
print("coarse_group: ", conv_0_layer.coarse_group)
print("fine: ", conv_0_layer.fine)

print("Latency (cycle):", conv_0_layer.latency())
print(conv_0_layer.resource())

coarse_in:  3
coarse_out:  16
coarse_group:  1
fine:  9
Latency (cycle): 1156
{'LUT': 18032, 'FF': 25236, 'DSP': 432, 'BRAM': 195, 'URAM': 0}


# 4.1_optimiser_transform

We load the parsed IR (and partition, if any) and read the platform/board settings, then pick the optimisation objective (latency, throughput, or balanced).
This helps anchor every transform choice to the exact model and device we’re targeting.

In [33]:
parser = Parser(custom_onnx=True, batch_size=1)
net = parser.onnx_to_fpgaconvnet(onnx_path)

for i, partition in enumerate(net.partitions):
    print(f"{i}, resource: ", partition.get_resource_usage())

CRITICAL WARNING: node Flatten_98 is skipped in hardware
0, resource:  {'FF': 76310, 'LUT': 92878, 'DSP': 55, 'BRAM': 1192, 'URAM': 0}


We define the transform knobs: coarse/fine folding factors, weight-reloading policy, precision/fixed-point format, batching, and any hard resource/bandwidth limits.
This helps make the search space explicit so later steps only consider hardware-valid designs.

In [34]:
transforms.partition.split_complete(net, None)
for i, partition in enumerate(net.partitions):
    transforms.weights_reloading.remove_weights_reloading_transform(partition)
    partition.update()
    print(f"{i}, resource: ", partition.get_resource_usage())

0, resource:  {'FF': 2094, 'LUT': 4140, 'DSP': 1, 'BRAM': 2, 'URAM': 0}
1, resource:  {'FF': 35, 'LUT': 16, 'DSP': 0, 'BRAM': 0, 'URAM': 0}
2, resource:  {'FF': 2133, 'LUT': 4120, 'DSP': 1, 'BRAM': 3, 'URAM': 0}
3, resource:  {'FF': 35, 'LUT': 16, 'DSP': 0, 'BRAM': 0, 'URAM': 0}
4, resource:  {'FF': 248, 'LUT': 267, 'DSP': 1, 'BRAM': 1, 'URAM': 0}
5, resource:  {'FF': 259, 'LUT': 323, 'DSP': 1, 'BRAM': 2, 'URAM': 0}
6, resource:  {'FF': 35, 'LUT': 16, 'DSP': 0, 'BRAM': 0, 'URAM': 0}
7, resource:  {'FF': 2176, 'LUT': 4111, 'DSP': 1, 'BRAM': 9, 'URAM': 0}
8, resource:  {'FF': 35, 'LUT': 16, 'DSP': 0, 'BRAM': 0, 'URAM': 0}
9, resource:  {'FF': 250, 'LUT': 278, 'DSP': 1, 'BRAM': 1, 'URAM': 0}
10, resource:  {'FF': 2832, 'LUT': 4873, 'DSP': 3, 'BRAM': 28, 'URAM': 0}
11, resource:  {'FF': 273, 'LUT': 390, 'DSP': 1, 'BRAM': 3, 'URAM': 0}
12, resource:  {'FF': 35, 'LUT': 16, 'DSP': 0, 'BRAM': 0, 'URAM': 0}
13, resource:  {'FF': 2200, 'LUT': 4111, 'DSP': 1, 'BRAM': 12, 'URAM': 0}
14, resource: 

We apply these transforms to create a transformed IR/view (updating tile sizes, stream widths, buffer needs) while keeping tensor shapes/rates consistent across stages.
This helps turn abstract options into a concrete design variant that downstream tools can build.

In [35]:
for i, partition in enumerate(net.partitions):
    transforms.weights_reloading.apply_max_weights_reloading(partition)
    partition.update()
    print(f"{i}, resource: ", partition.get_resource_usage())

0, resource:  {'FF': 2090, 'LUT': 4120, 'DSP': 1, 'BRAM': 1, 'URAM': 0}
1, resource:  {'FF': 35, 'LUT': 16, 'DSP': 0, 'BRAM': 0, 'URAM': 0}
2, resource:  {'FF': 2133, 'LUT': 4120, 'DSP': 1, 'BRAM': 3, 'URAM': 0}
3, resource:  {'FF': 35, 'LUT': 16, 'DSP': 0, 'BRAM': 0, 'URAM': 0}
4, resource:  {'FF': 246, 'LUT': 258, 'DSP': 1, 'BRAM': 0, 'URAM': 0}
5, resource:  {'FF': 246, 'LUT': 258, 'DSP': 1, 'BRAM': 0, 'URAM': 0}
6, resource:  {'FF': 35, 'LUT': 16, 'DSP': 0, 'BRAM': 0, 'URAM': 0}
7, resource:  {'FF': 2176, 'LUT': 4111, 'DSP': 1, 'BRAM': 9, 'URAM': 0}
8, resource:  {'FF': 35, 'LUT': 16, 'DSP': 0, 'BRAM': 0, 'URAM': 0}
9, resource:  {'FF': 246, 'LUT': 258, 'DSP': 1, 'BRAM': 0, 'URAM': 0}
10, resource:  {'FF': 2832, 'LUT': 4873, 'DSP': 3, 'BRAM': 28, 'URAM': 0}
11, resource:  {'FF': 246, 'LUT': 258, 'DSP': 1, 'BRAM': 0, 'URAM': 0}
12, resource:  {'FF': 35, 'LUT': 16, 'DSP': 0, 'BRAM': 0, 'URAM': 0}
13, resource:  {'FF': 2200, 'LUT': 4111, 'DSP': 1, 'BRAM': 12, 'URAM': 0}
14, resource: 

We sweep or tune selected knobs (e.g., folding/PE counts) to generate a small set of candidate configurations for this objective.
This helps explore useful trade-offs without exploding the search space.

In [36]:
conv_0_layer = net.partitions[0].graph.nodes["Conv_0"]["hw"]

print("coarse_in: ", conv_0_layer.coarse_in)
print("coarse_out: ", conv_0_layer.coarse_out)
print("coarse_group: ", conv_0_layer.coarse_group)
print("fine: ", conv_0_layer.fine)
print("Latency (cycle):", conv_0_layer.latency())
print(conv_0_layer.resource())

coarse_in:  1
coarse_out:  1
coarse_group:  1
fine:  1
Latency (cycle): 27648
{'LUT': 4120, 'FF': 2090, 'DSP': 1, 'BRAM': 1, 'URAM': 0}


We run quick feasibility checks and estimates (ops per stage, buffers/BRAM, DSP/LUT usage, interface rate matching) and prune invalid candidates.
This helps catch impossible or obviously poor designs early, saving solver/HLS time.

In [37]:
while conv_0_layer.coarse_in * conv_0_layer.coarse_out * conv_0_layer.coarse_group == 1:
    transforms.apply_random_coarse_node(net.partitions[0], "Conv_0")
    net.partitions[0].update()

print("coarse_in: ", conv_0_layer.coarse_in)
print("coarse_out: ", conv_0_layer.coarse_out)
print("coarse_group: ", conv_0_layer.coarse_group)
print("fine: ", conv_0_layer.fine)
print("Latency (cycle):", conv_0_layer.latency())
print(conv_0_layer.resource())

coarse_in:  3
coarse_out:  1
coarse_group:  1
fine:  1
Latency (cycle): 9216
{'LUT': 6748, 'FF': 3303, 'DSP': 3, 'BRAM': 3, 'URAM': 0}


We save the chosen transform configuration(s) to disk and print a short summary of key settings and expected effects.
This helps keep runs reproducible and provides a clean input for the solver in 4.2.

In [38]:
transforms.apply_complete_fine(net.partitions[0])
net.partitions[0].update()

print("coarse_in: ", conv_0_layer.coarse_in)
print("coarse_out: ", conv_0_layer.coarse_out)
print("coarse_group: ", conv_0_layer.coarse_group)
print("fine: ", conv_0_layer.fine)
print("Latency (cycle):", conv_0_layer.latency())
print(conv_0_layer.resource())

coarse_in:  3
coarse_out:  1
coarse_group:  1
fine:  9
Latency (cycle): 1156
{'LUT': 2519, 'FF': 3113, 'DSP': 27, 'BRAM': 3, 'URAM': 0}


# 4.2_optimiser_solver

We load the transformed IR/partition, platform constraints (part, clock, port width), and the optimisation objective (e.g., minimise latency or maximise throughput).
This helps make sure the solver searches with the exact same model and device assumptions as previous steps.

In [39]:
parser = Parser(custom_onnx=True, batch_size=1)
net = parser.onnx_to_fpgaconvnet(onnx_path)

CRITICAL WARNING: node Flatten_98 is skipped in hardware


We define the solver variables and bounds (e.g., PE counts, folding factors, buffer depths) plus hard constraints (DSP/BRAM/LUT budgets, bandwidth, rate matching).
This helps tell the solver what it is allowed to change and what limits it must never violate.

In [40]:
transforms.partition.split_complete(net, None)
for i, partition in enumerate(net.partitions):
    transforms.weights_reloading.apply_max_weights_reloading(partition)
    partition.update()

We set up the cost model and scoring function (latency/throughput formulas, penalty terms for near-violations) and choose the search method (ILP/heuristic/grid).
This helps translate “best design” into numbers the solver can actually optimise.

In [41]:
platform = Platform()
platform.update(board)
# We select port width 64 in this example
platform.port_width = 64

We run the solver to explore candidate designs and capture solutions (configs + predicted metrics + resource estimates).
This helps generate concrete options rather than guesswork, ready for feasibility checks.

In [42]:
opt = SimulatedAnnealing(net, platform)
opt.bram_to_lut = False
opt.off_chip_streaming = False
opt.balance_bram_uram = False
opt.rsc_allocation = 0.75


opt.run_solver()
opt.update_partitions()

pathlib.Path(pathlib_sa).mkdir(parents=True, exist_ok=True)
opt.create_report(pathlib_sa_report)
opt.net.save_all_partitions(pathlib_sa_config)

create edge ('Add_89', 'Conv_90')
create edge ('Relu_34', 'Conv_35')
create edge ('Conv_90', 'Relu_91')
create edge ('Add_43', 'Add_43_split')
create edge ('GlobalAveragePool_97', 'Gemm_99')
create edge ('Conv_77', 'Conv_77_split')
create edge ('Conv_35', 'Relu_36')
create edge ('Relu_19', 'Conv_20')
create edge ('Add_15', 'Conv_16')
create edge ('Relu_96', 'GlobalAveragePool_97')
create edge ('Conv_73', 'Relu_74')
create edge ('Add_26', 'Add_26_split')
create edge ('Conv_60', 'Conv_60_split')
create edge ('Add_49', 'Add_49_split')
create edge ('Relu_36', 'Conv_37')
create branch edge ('Conv_31', 'Add_32')
create edge ('Add_32', 'Conv_33')
create edge ('Relu_76', 'Conv_77')
create branch edge ('Add_83_split', 'Add_89')
create branch edge ('Add_83_split', 'Conv_84')
create edge ('Add_83', 'Add_83_split')
create edge ('Conv_4', 'Conv_5')
create edge ('Add_26', 'Add_26_split')
create edge ('Relu_59', 'Conv_60')
create branch edge ('Add_26_split', 'Add_32')
create branch edge ('Add_26_spli

We filter infeasible candidates and select the best valid design according to the objective, then create a compact config (only the knobs downstream needs).
This helps ensure the chosen result really fits the device and is simple to consume by HLS.

In [43]:
net = parser.onnx_to_fpgaconvnet(onnx_path)
transforms.partition.split_complete(net, None)
for i, partition in enumerate(net.partitions):
    transforms.weights_reloading.apply_max_weights_reloading(partition)
    partition.update()

CRITICAL WARNING: node Flatten_98 is skipped in hardware


We save the winning configuration and a short summary/report to disk (JSON + printed table/metrics).
This helps keep runs reproducible and gives the next step (4.3 reporting and 5.x HLS) a clean, reusable input.

In [44]:
opt = GreedyPartition(net, platform)
opt.bram_to_lut = False
opt.off_chip_streaming = False
opt.balance_bram_uram = False
opt.rsc_allocation = 0.75
opt.transforms = []
opt.transforms_probs = []


opt.run_solver()
opt.update_partitions()
opt.merge_memory_bound_partitions()
opt.update_partitions()

pathlib.Path(pathlib_gp).mkdir(parents=True, exist_ok=True)
opt.create_report(pathlib_gp_report)
opt.net.save_all_partitions(pathlib_gp_config)

ERROR: Exceeds resource usage:
ERROR: Memory bandwidth exceeded, partition: 47
resolving memory bound partitions
||  Attemting Partition Merging:    ||  (Part 39, Part 40)    ||      ||  Total Partitions:    ||  49    ||
create edge ('Add_83', 'Add_83_split')
ERROR: Exceeds resource usage:
ERROR: BRAM usage exceeded, partition: 38
||  Outcome:    ||  Merge Rejected    ||
||  Attemting Partition Merging:    ||  (Part 38, Part 39)    ||      ||  Total Partitions:    ||  49    ||
create edge ('Conv_77', 'Conv_77_split')
ERROR: Exceeds resource usage:
ERROR: Memory bandwidth exceeded, partition: 46
||  Outcome:    ||  Merge Rejected    ||
||  Attemting Partition Merging:    ||  (Part 40, Part 41)    ||      ||  Total Partitions:    ||  49    ||
create edge ('Add_89', 'Conv_90')
ERROR: Exceeds resource usage:
ERROR: Memory bandwidth exceeded, partition: 46
||  Outcome:    ||  Merge Rejected    ||
||  Attemting Partition Merging:    ||  (Part 32, Part 33)    ||      ||  Total Partitions:    

# 4.3_optimiser_config_report

We load all artefacts from previous steps (IR/partition, transform settings, solver’s best config, platform info) and verify the files/paths exist.
This helps ensure the report is built from the exact design we just optimised and avoids broken or stale references.

In [45]:
with open(pathlib_sa_config) as f:
    config = json.load(f)
display(JSON(config))

<IPython.core.display.JSON object>

We collate key metrics into a single summary object/table: clock, port width, predicted latency/throughput, and resource estimates (DSP/BRAM/LUT).
This helps give a clear, human-readable snapshot of the chosen design point in one place.

In [46]:
with open(pathlib_sa_report) as f:
    report = json.load(f)
display(JSON(report))

<IPython.core.display.JSON object>

We add simple derived stats and highlights (e.g., FPS from latency, utilisation ratios, largest tensors/stages) and flag any obvious bottlenecks.
This helps interpret numbers at a glance so readers understand trade-offs without digging into raw logs.

In [47]:
parser = Parser(custom_onnx=True, batch_size=1)
net = parser.onnx_to_fpgaconvnet(onnx_path)
net = parser.prototxt_to_fpgaconvnet(net, pathlib_sa_config)

platform = Platform()
platform.update(board)

solver = Solver(net, platform)
solver.solver_status()

CRITICAL WARNING: node Flatten_98 is skipped in hardware
| COST:                 |    | RESOURCES:   |            |                 |                 |             |             |             |             |
|-----------------------|----|--------------|------------|-----------------|-----------------|-------------|-------------|-------------|-------------|
|                       |    | BRAM         | DSP        | LUT             | FF              | BW          | BW_IN       | BW_OUT      | BW_WEIGHT   |
| 0.166905 (throughput) |    | 250.31/280   | 327.38/220 | 186758.38/53200 | 67741.38/106400 | 8.56/34.112 | 4.22/34.112 | 4.34/34.112 | 0.00/34.112 |
|                       |    | 89.40 %      | 148.81 %   | 351.05 %        | 63.67 %         | 25.10 %     | 12.36 %     | 12.74 %     | 0.00 %      |



We write the report to disk (JSON/CSV/Markdown) under reports/, preserving paths to all inputs (ONNX, configs) for traceability.
This helps keep runs reproducible and lets downstream steps or teammates load the same configuration.

In [48]:
print("Latency: ", report['network']['performance']['latency (s)'])
print("Throughput: ", report['network']['performance']['throughput (FPS)'])

Latency:  5.99159979
Throughput:  0.1669003329743424


We print a short end-of-section digest (file locations, sizes, checksums) and list recommended next steps (HLS generation, C-sim, Co-sim).
This helps confirm everything is in place and guides the reader to the hardware build stage.

In [49]:
print("Resource: ", report['network']['avg_resource_usage'])

Resource:  {'LUT': 186758.38461538462, 'FF': 67741.38461538461, 'BRAM': 587.6153846153846, 'DSP': 327.38461538461536}


# 5_hls

We instantiate the HLS generator (e.g., GenerateNetwork) with the network name, config path, ONNX path, device, clock period, and port width.
This helps centralize all build parameters so every downstream HLS step is driven by one consistent object.

In [50]:
# Any final changes can be done in this stage
network_name = model_name
config_path  = pathlib_gp_config       # keep your greedy-partition config
onnx_path    = onnx_path               # keep the same ONNX you selected
device_path = board

with open(device_path, "r") as f:
    device = toml.load(f)
    device_name = device["device"]["part"]
    clock_period = 1000 / device["system"]["board_frequency"]
    # port_width = device["system"]["port_width"]

# Same as optimiser-solver, select port_width = 64 for HLS
port_width = 64

# create instance of the network
gen_net = GenerateNetwork(network_name, config_path, onnx_path, device_name, clock_period, port_width)

FIXME: use HLS backend in Parser
CRITICAL WARNING: node Flatten_98 is skipped in hardware


We create the Vivado HLS project for the chosen partition/stage and populate sources, headers, and scripts.
This helps scaffold a reproducible project structure that HLS can synthesize without manual setup.

In [51]:
gen_net.create_partition_project(0)

--------- Generating layer Conv_0 ---------
Writing weights stream to .dat file
Streams:1, Ports:1, PortWidth:64, DataWidth:16, Maximum stream number: 4.0, Number of elements = 432
Writing biases stream to .dat file
Streams:1, Ports:1, PortWidth:64, DataWidth:32, Maximum stream number: 2.0, Number of elements = 16

****** Vivado(TM) HLS - High-Level Synthesis from C, C++ and SystemC v2019.1 (64-bit)
  **** SW Build 2552052 on Fri May 24 14:47:09 MDT 2019
  **** IP Build 2548770 on Fri May 24 18:01:18 MDT 2019
    ** Copyright 1986-2019 Xilinx, Inc. All Rights Reserved.

source /home/ubuntu/Xilinx/Vivado/2019.1/scripts/vivado_hls/hls.tcl -notrace
INFO: Applying HLS Y2K22 patch v1.2 for IP revision
INFO: [HLS 200-10] Running '/home/ubuntu/Xilinx/Vivado/2019.1/bin/unwrapped/lnx64.o/vivado_hls'
INFO: [HLS 200-10] For user 'ubuntu' on host 'ubuntu-ROG-Zephyrus-Duo-16-GX650PY-GX650PY' (Linux_x86_64 version 6.14.0-36-generic) on Wed Dec 10 12:22:44 EET 2025
INFO: [HLS 200-10] On os Ubuntu 24.

We trigger code generation and C-synthesis for the partition to produce RTL and utilization/timing estimates.
This helps turn our high-level design into hardware and quickly check whether the target clock and resources look feasible.

In [52]:
# create random input image
input_image = np.random.rand(1,3,32,32).astype(np.float32)
# run C simulation
gen_net.run_testbench(0, input_image)

Writing input stream to .dat file
Streams:1, Ports:1, PortWidth:64, DataWidth:16, Maximum stream number: 4.0, Number of elements = 3072
Writing valid output stream to .dat file
Streams:1, Ports:1, PortWidth:64, DataWidth:16, Maximum stream number: 4.0, Number of elements = 16384

****** Vivado(TM) HLS - High-Level Synthesis from C, C++ and SystemC v2019.1 (64-bit)
  **** SW Build 2552052 on Fri May 24 14:47:09 MDT 2019
  **** IP Build 2548770 on Fri May 24 18:01:18 MDT 2019
    ** Copyright 1986-2019 Xilinx, Inc. All Rights Reserved.

source /home/ubuntu/Xilinx/Vivado/2019.1/scripts/vivado_hls/hls.tcl -notrace
INFO: Applying HLS Y2K22 patch v1.2 for IP revision
INFO: [HLS 200-10] Running '/home/ubuntu/Xilinx/Vivado/2019.1/bin/unwrapped/lnx64.o/vivado_hls'
INFO: [HLS 200-10] For user 'ubuntu' on host 'ubuntu-ROG-Zephyrus-Duo-16-GX650PY-GX650PY' (Linux_x86_64 version 6.14.0-36-generic) on Wed Dec 10 12:22:46 EET 2025
INFO: [HLS 200-10] On os Ubuntu 24.04.3 LTS
INFO: [HLS 200-10] In direc

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



We run the C-level testbench with a dummy input (or prepared sample) to verify functional correctness before RTL.
This helps catch obvious logic or interface issues early, when they’re fastest to fix.

In [53]:
# run C synthesis and export IP for partition 0
gen_net.generate_partition_hardware(0)


****** Vivado(TM) HLS - High-Level Synthesis from C, C++ and SystemC v2019.1 (64-bit)
  **** SW Build 2552052 on Fri May 24 14:47:09 MDT 2019
  **** IP Build 2548770 on Fri May 24 18:01:18 MDT 2019
    ** Copyright 1986-2019 Xilinx, Inc. All Rights Reserved.

source /home/ubuntu/Xilinx/Vivado/2019.1/scripts/vivado_hls/hls.tcl -notrace
INFO: Applying HLS Y2K22 patch v1.2 for IP revision
INFO: [HLS 200-10] Running '/home/ubuntu/Xilinx/Vivado/2019.1/bin/unwrapped/lnx64.o/vivado_hls'
INFO: [HLS 200-10] For user 'ubuntu' on host 'ubuntu-ROG-Zephyrus-Duo-16-GX650PY-GX650PY' (Linux_x86_64 version 6.14.0-36-generic) on Wed Dec 10 12:22:56 EET 2025
INFO: [HLS 200-10] On os Ubuntu 24.04.3 LTS
INFO: [HLS 200-10] In directory '/home/ubuntu/fpgaConvNet/fpgaconvnet-tutorial/8_All_In_One_ONNX'
Sourcing Tcl script '/home/ubuntu/fpgaConvNet/fpgaconvnet-hls/fpgaconvnet/hls/scripts/hls/run_csynth.tcl'
INFO: [HLS 200-10] Opening project '/home/ubuntu/fpgaConvNet/fpgaconvnet-tutorial/8_All_In_One_ONNX/par

We run RTL co-simulation using the same input to cross-check C vs. RTL behavior.
This helps ensure the synthesized RTL matches the algorithmic model and is safe to package.

In [54]:
# run cosimulation
gen_net.run_cosimulation(0, input_image)

Writing input stream to .dat file
Streams:1, Ports:1, PortWidth:64, DataWidth:16, Maximum stream number: 4.0, Number of elements = 3072
Writing valid output stream to .dat file
Streams:1, Ports:1, PortWidth:64, DataWidth:16, Maximum stream number: 4.0, Number of elements = 16384

****** Vivado(TM) HLS - High-Level Synthesis from C, C++ and SystemC v2019.1 (64-bit)
  **** SW Build 2552052 on Fri May 24 14:47:09 MDT 2019
  **** IP Build 2548770 on Fri May 24 18:01:18 MDT 2019
    ** Copyright 1986-2019 Xilinx, Inc. All Rights Reserved.

source /home/ubuntu/Xilinx/Vivado/2019.1/scripts/vivado_hls/hls.tcl -notrace
INFO: Applying HLS Y2K22 patch v1.2 for IP revision
INFO: [HLS 200-10] Running '/home/ubuntu/Xilinx/Vivado/2019.1/bin/unwrapped/lnx64.o/vivado_hls'
INFO: [HLS 200-10] For user 'ubuntu' on host 'ubuntu-ROG-Zephyrus-Duo-16-GX650PY-GX650PY' (Linux_x86_64 version 6.14.0-36-generic) on Wed Dec 10 12:24:02 EET 2025
INFO: [HLS 200-10] On os Ubuntu 24.04.3 LTS
INFO: [HLS 200-10] In direc

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



INFO: [COSIM 212-333] Generating C post check test bench ...
INFO: [COSIM 212-12] Generating RTL test bench ...
INFO: [COSIM 212-1] *** C/RTL co-simulation file generation completed. ***
INFO: [COSIM 212-323] Starting verilog simulation. 
INFO: [COSIM 212-15] Starting XSIM ...
INFO: [XSIM 43-3496] Using init file passed via -initfile option "/home/ubuntu/Xilinx/Vivado/2019.1/data/xsim/ip/xsim_ip.ini".
Vivado Simulator 2019.1
Copyright 1986-1999, 2001-2019 Xilinx, Inc. All Rights Reserved.
Running: /home/ubuntu/Xilinx/Vivado/2019.1/bin/unwrapped/lnx64.o/xelab xil_defaultlib.apatb_fpgaconvnet_ip_top glbl -prj fpgaconvnet_ip.prj -L smartconnect_v1_0 -L axi_protocol_checker_v1_1_12 -L axi_protocol_checker_v1_1_13 -L axis_protocol_checker_v1_1_11 -L axis_protocol_checker_v1_1_12 -L xil_defaultlib -L unisims_ver -L xpm --initfile /home/ubuntu/Xilinx/Vivado/2019.1/data/xsim/ip/xsim_ip.ini --lib ieee_proposed=./ieee_proposed -s fpgaconvnet_ip -debug wave 
Multi-threading is on. Using 30 slave 

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



We move everything to a compact folder. 
This helps ensure that we can do multiple runs using multiple models and to know which model corresponds to each board.

In [55]:
import shutil
from pathlib import Path

# We assume you run the notebook from /home/ubuntu/Xilinx/Projects
project_root = Path.cwd()

# Name of THIS notebook (so we can archive it too)
# 🔴 Change this if you rename the .ipynb file
notebook_name = "8_All_In_One_ONNX.ipynb"

# 'model_name' and 'board' already exist in your notebook
# board is something like "ProjectFiles/platforms/zedboard.toml"
board_name = Path(board).stem

# 'model_status' is to label whether the model is complete, partial or not working
# possible names: "Working" , "Partial" , "Not Working"
model_status = "Partial"

# Destination: ProjectFiles/hls_builds/model--board--status
dest_root = project_root / "ProjectFiles" / "hls_builds" / f"{board_name}--{model_name}--{model_status}"
dest_root.mkdir(parents=True, exist_ok=True)

print(f"Collecting artefacts into: {dest_root}")

# ============================================================
# 1) Move partition_* directories
# ============================================================
for p in project_root.glob("partition_*"):
    if p.is_dir():
        target = dest_root / p.name
        if target.exists():
            # If you re-run, overwrite old results for this partition
            shutil.rmtree(target)
        shutil.move(str(p), str(target))
        print(f"  - moved {p.name}")

# ============================================================
# 2) Move Vivado HLS log if present
# ============================================================
log_file = project_root / "vivado_hls.log"
if log_file.exists():
    target = dest_root / log_file.name
    if target.exists():
        target.unlink()
    shutil.move(str(log_file), str(target))
    print("  - moved vivado_hls.log")
else:
    print("  - vivado_hls.log not found (skipping)")

# ============================================================
# 3) Copy optimisation reports (SA / GP) into this results folder
#    Uses your existing:
#    pathlib_sa, pathlib_gp
# ============================================================
reports_dest_root = dest_root / "reports"
reports_dest_root.mkdir(exist_ok=True)

def copy_report_folder(src_dir_str, subdir_name, label):
    src_dir = project_root / src_dir_str
    dest_dir = reports_dest_root / subdir_name

    if src_dir.exists():
        if dest_dir.exists():
            shutil.rmtree(dest_dir)
        shutil.copytree(src_dir, dest_dir)
        print(f"  - copied {label} reports: {src_dir} -> {dest_dir}")
    else:
        print(f"  - WARNING: {label} report folder not found at {src_dir}")

# SA reports (Simulated Annealing)
copy_report_folder(pathlib_sa, "sa", "SA")

# GP reports (Greedy Partition)
copy_report_folder(pathlib_gp, "gp", "GP")

# ============================================================
# 5) Copy THIS NOTEBOOK (.ipynb) into the results folder
#    So you keep all cell outputs, runtimes, logs, etc.
# ============================================================
notebook_path = project_root / notebook_name
if notebook_path.exists():
    dest_notebook = dest_root / notebook_path.name
    shutil.copy2(notebook_path, dest_notebook)
    print(f"  - copied notebook: {notebook_path} -> {dest_notebook}")
else:
    print(f"  - WARNING: notebook {notebook_name} not found in {project_root}")


  - moved partition_3
  - moved partition_21
  - moved partition_45
  - moved partition_24
  - moved partition_43
  - moved partition_32
  - moved partition_34
  - moved partition_8
  - moved partition_17
  - moved partition_11
  - moved partition_40
  - moved partition_7
  - moved partition_15
  - moved partition_2
  - moved partition_28
  - moved partition_16
  - moved partition_37
  - moved partition_47
  - moved partition_22
  - moved partition_38
  - moved partition_1
  - moved partition_0
  - moved partition_19
  - moved partition_39
  - moved partition_5
  - moved partition_18
  - moved partition_26
  - moved partition_25
  - moved partition_27
  - moved partition_29
  - moved partition_20
  - moved partition_44
  - moved partition_31
  - moved partition_42
  - moved partition_9
  - moved partition_46
  - moved partition_6
  - moved partition_41
  - moved partition_12
  - moved partition_10
  - moved partition_33
  - moved partition_23
  - moved partition_13
  - moved partition_